# 6.9 Gradient Boosting

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.09_gradient-boosting.ipynb)

Companion code for Section 6.9 of the book. Run the cells in order from top to bottom.

**Example 6.9.1**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

# ── Standard Gradient Boosting ────────────────────────────────────
print('=== Gradient Boosting Hyperparameter Analysis ===')
print(f'{'Config':<35} {'Train Acc':>10} {'Test Acc':>10} {'AUC':>8}')
print('='*67)

configs = [
    ('n=100, lr=0.1, depth=3',  {'n_estimators':100, 'learning_rate':0.1, 'max_depth':3}),
    ('n=100, lr=0.01, depth=3', {'n_estimators':100, 'learning_rate':0.01,'max_depth':3}),
    ('n=200, lr=0.1, depth=3',  {'n_estimators':200, 'learning_rate':0.1, 'max_depth':3}),
    ('n=100, lr=0.1, depth=5',  {'n_estimators':100, 'learning_rate':0.1, 'max_depth':5}),
    ('n=500, lr=0.05, depth=3', {'n_estimators':500, 'learning_rate':0.05,'max_depth':3}),
]

for name, params in configs:
    gb = GradientBoostingClassifier(**params, random_state=42)
    gb.fit(X_train, y_train)
    tr = gb.score(X_train, y_train)
    te = gb.score(X_test, y_test)
    auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:,1])
    print(f'{name:<35} {tr:>10.4f} {te:>10.4f} {auc:>8.4f}')

# ── HistGradientBoosting (much faster for large datasets) ─────────
print('\n=== HistGradientBoostingClassifier (sklearn fast implementation) ===')
hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                      max_depth=3, random_state=42)
hgb.fit(X_train, y_train)
print(f'Test accuracy: {hgb.score(X_test, y_test):.4f}')
print(f'AUC:           {roc_auc_score(y_test, hgb.predict_proba(X_test)[:,1]):.4f}')

# ── Staged predictions - how accuracy improves with more trees ─────
best_gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                      max_depth=3, random_state=42)
best_gb.fit(X_train, y_train)

train_staged = [acc for acc in best_gb.staged_score(X_train, y_train)]
test_staged  = [acc for acc in best_gb.staged_score(X_test,  y_test)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_staged, 'b-', lw=2, alpha=0.8, label='Train')
ax.plot(test_staged,  'r-', lw=2, alpha=0.8, label='Test')
ax.axvline(np.argmax(test_staged), color='green', ls='--',
          label=f'Best: {np.argmax(test_staged)+1} trees')
ax.set_xlabel('Number of Boosting Rounds'); ax.set_ylabel('Accuracy')
ax.set_title('Gradient Boosting: Staged Accuracy (Train vs Test)',
            fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gradient_boosting.png', dpi=150, bbox_inches='tight')
plt.show()